# Bônus: Mapas Contrafactuais

**Pergunta:** E a evidência *contra*?

Este notebook demonstra o mapa contrafactual, obtido ao negar os pesos $\alpha$ antes da aplicação do ReLU, deslocando o foco para evidências contrárias à classe.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('.'))
sys.path.append(os.path.abspath('notebooks'))
import deck
import torch
import matplotlib.pyplot as plt
from PIL import Image
from gradcam.models import load_model, preprocess
from gradcam.core import gradcam
from gradcam.viz import scale_for_display, overlay, gray_rgb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model("vgg16", device=device)
model.eval()
print("Ambiente carregado para Bônus (Contrafactual).")

## Calculando o Grad-CAM Contrafactual

Invertendo os pesos para evidenciar o que **não** pertence à classe.

In [ ]:
base_dir = os.path.abspath('.')
if os.path.basename(base_dir) == 'notebooks':
    img_path = "assets/cat_dog.jpg"
else:
    img_path = "notebooks/assets/cat_dog.jpg"
pil_img = Image.open(img_path).convert("RGB")
tensor_img = preprocess(pil_img).to(device)

# Grad-CAM sem ReLU com alpha negado para contrafactual
gcam_neg, class_idx = gradcam(model, tensor_img, class_idx=None, layer=None, apply_relu=False)
gcam_neg = (-gcam_neg).clamp(min=0)
gcam_neg_scaled = scale_for_display(gcam_neg, high=deck.DISPLAY_HIGH, gamma=deck.DISPLAY_GAMMA)

fig, ax = plt.subplots(figsize=(5, 5), dpi=110)
ax.imshow(gray_rgb(gcam_neg_scaled))
ax.axis("off")
ax.set_title(f"Grad-CAM Contrafactual (Classe ID: {class_idx})")
deck.show(fig)